# Merging ENTSO-E download ranges

The dataset was first downloaded for `2022-12-01_2026-09-16`. It was then extended with two extra downloads, so three ranges now exist side by side under `data/raw/entsoe/<zone>/<range>/`:

* `2022-06-09_2023-01-01` — extends the start,
* `2022-12-01_2026-09-16` — the original download,
* `2026-09-01_2026-10-21` — extends the end.

This notebook merges them into one continuous range, `DOWNLOAD_START_DATE_DOWNLOAD_END_DATE` from `src/config.py`, with the same folder layout as the fetching step, so the next stages can read it unchanged.

**Rules**

* **Zones.** Only zones that have all three range folders are merged. Zones present in just the original range (for example PL, which was not downloaded for the extensions) are skipped.
* **Datasets.** Only datasets present in all three ranges. The unavailability files exist only in the original range, so they are left out.
* **Columns.** Only columns present in every range. A single unnamed column (`0`, from the original day-ahead price files) is renamed to the dataset name.
* **Overlaps.** The ranges overlap, so rows are merged by index. Where a timestamp occurs more than once, the first range in `source_ranges` wins. Timestamps repeated inside one file, from the month boundaries of the batched download, are dropped the same way.
* **Checks.** The index must be unique and increasing, the overlaps must hold identical values, and the merge must not open any gap between the ranges.

In [1]:
import sys
import logging
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import DATA_DIR, DOWNLOAD_START_DATE, DOWNLOAD_END_DATE
from src.utils.file_io import inspect_merged_file, save_entsoe_data
from src.utils.range_merging import merge_entsoe_ranges, scan_zones

logging.basicConfig(level=logging.INFO)

In [2]:
raw_dir = DATA_DIR / "raw" / "entsoe"

source_ranges = [
    "2022-12-01_2026-09-16",
    "2022-06-09_2023-01-01",
    "2026-09-01_2026-10-21",
]
output_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

assert output_range not in source_ranges

## 1. Zones and Datasets

`complete` is true for a zone that has every range folder. `datasets` lists what is merged for it, and `not_in_all_ranges` what is skipped.

In [3]:
zones = scan_zones(raw_dir, source_ranges)
zones

,ranges_found,complete,datasets,not_in_all_ranges
zone,,,,
AT,1,False,[],[]
CZ,3,True,"[day_ahead_prices, generation_forecast, genera...",[]
DE_LU,3,True,"[day_ahead_prices, generation_forecast, genera...","[generation_units_unavailability, production_u..."
FR,1,False,[],[]
HU,1,False,[],[]
LT,3,True,"[day_ahead_prices, generation_forecast, genera...",[]
NO,1,False,[],[]
PL,3,True,"[day_ahead_prices, generation, generation_fore...","[generation_units_unavailability, production_u..."
SE,1,False,[],[]


## 2. Merge

In [4]:
merged, report = merge_entsoe_ranges(raw_dir, zones, source_ranges)
report.drop(columns=["unique", "monotonic"])

,zone,dataset,rows,start,end,duplicates_removed,overlap_mismatches,dropped_columns,gaps,longest_gap,seam_gaps
0,CZ,day_ahead_prices,65998,2022-06-09 00:00:00+02:00,2026-10-21 00:00:00+02:00,2193,0,[],2,0 days 02:00:00,0
1,CZ,generation_forecast,98970,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,2184,0,[],0,0 days 00:00:00,0
2,CZ,generation_wind_solar_forecast,98970,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,2184,0,[],0,0 days 00:00:00,0
3,CZ,load_forecast,98970,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,2184,0,[],0,0 days 00:00:00,0
4,DE_LU,day_ahead_prices,65998,2022-06-09 00:00:00+02:00,2026-10-21 00:00:00+02:00,2193,0,[],2,0 days 02:00:00,0
5,DE_LU,generation_forecast,85716,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,2184,0,[],3,0 days 02:00:00,0
6,DE_LU,generation_wind_solar_forecast,153120,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,4416,0,[],0,0 days 00:00:00,0
7,DE_LU,load_forecast,153112,2022-06-09 00:00:00+02:00,2026-10-20 23:45:00+02:00,4416,0,[],2,0 days 01:15:00,0
8,LT,day_ahead_prices,65998,2022-06-09 01:00:00+03:00,2026-10-21 01:00:00+03:00,2193,0,[],2,0 days 02:00:00,0
9,LT,generation_forecast,90610,2022-06-09 01:00:00+03:00,2026-10-21 00:45:00+03:00,2184,0,[Actual Aggregated],5,8 days 01:15:00,0


## 3. Checks

`gaps` counts steps longer than one hour. They are already in the raw files, for example the night of a time change or a missing day in a forecast, and are handled in the cleaning and preprocessing stages. `seam_gaps` counts only gaps that the merge itself would have opened between the ranges.

In [5]:
assert report["unique"].all() and report["monotonic"].all()
assert (report["overlap_mismatches"] == 0).all()
assert (report["seam_gaps"] == 0).all()

print(
    f"{len(report)} datasets in {len(merged)} zones: index unique and increasing, "
    "overlaps identical, no gap at the seams"
)

25 datasets in 6 zones: index unique and increasing, overlaps identical, no gap at the seams


## 4. Saving Merged Data

Files go to `data/raw/entsoe/<zone>/<output_range>/<dataset>.parquet`, and each file is verified by reading it back.

In [6]:
save_entsoe_data(merged, raw_dir, output_range)

INFO:src.utils.file_io:Saved raw/entsoe/CZ/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:src.utils.file_io:Saved raw/entsoe/CZ/2022-06-09_2026-10-21/generation_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/CZ/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/CZ/2022-06-09_2026-10-21/load_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/DE_LU/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:src.utils.file_io:Saved raw/entsoe/DE_LU/2022-06-09_2026-10-21/generation_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/DE_LU/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/DE_LU/2022-06-09_2026-10-21/load_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/LT/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:src.utils.file_io:Saved raw/entsoe/LT/2022-06-09_2026-10-21/generation_forecast.parquet
INFO:src.utils.file_io:Saved raw/entsoe/LT/2022-06-09_202

In [7]:
for zone, datasets in merged.items():
    print(f"\n--- {zone} ({len(datasets)} file/s)")
    for name in datasets:
        inspect_merged_file(raw_dir / zone / output_range / f"{name}.parquet")


--- CZ (4 file/s)
  [FILE] data/raw/entsoe/CZ/2022-06-09_2026-10-21/day_ahead_prices.parquet | shape: (65998, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-21 00:00:00+02:00 | cols: 1 | NaN: 0
  [FILE] data/raw/entsoe/CZ/2022-06-09_2026-10-21/generation_forecast.parquet | shape: (98970, 2) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00 | cols: 2 | NaN: 0
  [FILE] data/raw/entsoe/CZ/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet | shape: (98970, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00 | cols: 1 | NaN: 0
  [FILE] data/raw/entsoe/CZ/2022-06-09_2026-10-21/load_forecast.parquet | shape: (98970, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00 | cols: 1 | NaN: 0

--- DE_LU (4 file/s)
  [FILE] data/raw/entsoe/DE_LU/2022-06-09_2026-10-21/day_ahead_prices.parquet | shape: (65998, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-21 00:00:00+02:00 | cols: 1 | NaN: 0
  [FILE] data/raw/entsoe/DE_LU/2022-06-09_2026-10-21/generation_forecast.parq